# Track A — S24 SD LoRA VS Generation (Colab)

**전제조건**:
- 런타임 유형: GPU (T4 이상 권장)
- Google Drive에 S24 VS 데이터 업로드 완료

**Google Drive 업로드 필요 파일** (NPU 서버에서):
```bash
# 서버에서 Drive로 업로드 (rclone 또는 수동)
# preproc_vs_re/preproc_subj_24_*.mat  (10개)
# → MyDrive/vsvi_data/preproc_vs_re/ 에 위치
```

**순서**: 셀을 위에서 아래로 순서대로 실행

In [ ]:
# [1] GPU 확인
import torch
print(f'torch {torch.__version__}')
print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
else:
    raise RuntimeError('GPU 런타임이 아닙니다. 런타임 > 런타임 유형 변경 > GPU 선택')

In [ ]:
# [2] Google Drive 마운트
from google.colab import drive
drive.mount('/content/drive')

# Drive 내 데이터 경로 확인
import os
DRIVE_DATA = '/content/drive/MyDrive/vsvi_data'
mat_files = [f for f in os.listdir(f'{DRIVE_DATA}/preproc_vs_re') if 'subj_24' in f]
print(f'S24 .mat files found: {len(mat_files)}')
for f in sorted(mat_files):
    print(f'  {f}')

In [ ]:
# [3] 코드 clone
import os
REPO = 'https://github.com/heegyukim4043/test_diffusion_EEG_visualstim_image.git'
WORKDIR = '/content/vsvi_project'

if not os.path.exists(WORKDIR):
    os.system(f'git clone {REPO} {WORKDIR}')
else:
    os.system(f'cd {WORKDIR} && git pull origin main')

os.chdir(WORKDIR)
print(f'Working dir: {os.getcwd()}')
os.system('git log --oneline -3')

In [ ]:
# [4] 패키지 설치
os.system('pip install -q peft diffusers accelerate transformers scipy')

import peft, diffusers
print(f'peft {peft.__version__}, diffusers {diffusers.__version__}')

In [ ]:
# [5] 데이터/체크포인트 경로 연결 (symlink)
import subprocess

DRIVE_DATA = '/content/drive/MyDrive/vsvi_data'
WORKDIR = '/content/vsvi_project'

# preproc_vs_re symlink
src = f'{DRIVE_DATA}/preproc_vs_re'
dst = f'{WORKDIR}/preproc_vs_re'
if not os.path.exists(dst):
    os.symlink(src, dst)
print(f'preproc_vs_re -> {src}')

# checkpoint output 디렉토리 (Drive에 저장)
ckpt_out = f'{DRIVE_DATA}/checkpoints_vsre_lora_gen'
os.makedirs(ckpt_out, exist_ok=True)
dst_ckpt = f'{WORKDIR}/checkpoints_vsre_lora_gen'
if not os.path.exists(dst_ckpt):
    os.symlink(ckpt_out, dst_ckpt)
print(f'checkpoints_vsre_lora_gen -> {ckpt_out}')

# 확인
print('\n--- Path check ---')
print('S24 mat files:', len([f for f in os.listdir(f'{WORKDIR}/preproc_vs_re') if 'subj_24' in f]))
print('class images:', os.path.exists(f'{WORKDIR}/preproc_data_vi/images/01.png'))
print('S24 SupCon ckpt:', os.path.exists(
    f'{WORKDIR}/checkpoints_vsre_dino/20260604_091352_ch32_merged_ep200_supcon/subj24_best.pt'))

In [ ]:
# [6] Preflight
os.chdir(WORKDIR)
ret = os.system(
    'python preflight_track_a.py '
    '--subject_id 24 '
    '--img_root preproc_data_vi/images '
    '--supcon_ckpt checkpoints_vsre_dino/20260604_091352_ch32_merged_ep200_supcon '
    '--data_root preproc_vs_re '
    '--ckpt_root checkpoints_vsre_lora_gen '
    '--check_data'
)
assert ret == 0, 'Preflight FAILED — 위 출력 확인 후 문제 해결'

In [ ]:
# [7] S24 LoRA r=16 학습
# 예상 소요시간: T4 기준 약 8~12시간
# Colab Pro 권장 (세션 유지 필요)
import subprocess, time

os.chdir(WORKDIR)
cmd = [
    'python', 'train_vs_re_lora_gen.py',
    '--subject_ids', '24',
    '--lora_r', '16',
    '--n_eeg_tokens', '16',
    '--epochs', '100',
    '--img_root', 'preproc_data_vi/images',
    '--supcon_ckpt', 'checkpoints_vsre_dino/20260604_091352_ch32_merged_ep200_supcon',
    '--ckpt_root', 'checkpoints_vsre_lora_gen',
]
print('Command:', ' '.join(cmd))
print('Starting...')
result = subprocess.run(cmd)
print(f'\nExit code: {result.returncode}')

In [ ]:
# [8] S24 LoRA r=32 학습 (r=16 완료 후 실행)
os.chdir(WORKDIR)
cmd = [
    'python', 'train_vs_re_lora_gen.py',
    '--subject_ids', '24',
    '--lora_r', '32',
    '--n_eeg_tokens', '16',
    '--epochs', '100',
    '--img_root', 'preproc_data_vi/images',
    '--supcon_ckpt', 'checkpoints_vsre_dino/20260604_091352_ch32_merged_ep200_supcon',
    '--ckpt_root', 'checkpoints_vsre_lora_gen',
]
print('Command:', ' '.join(cmd))
result = subprocess.run(cmd)
print(f'\nExit code: {result.returncode}')

In [ ]:
# [9] 결과 확인
import glob
results = glob.glob(f'{ckpt_out}/**/results_dual_gallery.csv', recursive=True)
for r in sorted(results):
    print(r)
    with open(r) as f:
        print(f.read())